<a href="https://colab.research.google.com/github/Lynna5639582/Revision-Templates/blob/main/PyTorch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Imports & Data Prep

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

# 1. Load & clean
df = pd.read_csv('YOUR_FILE.csv').dropna()  # <-- EDIT: filename
# df = pd.get_dummies(df, drop_first=True, dtype=int)  # <-- Uncomment if text columns exist

target_col = 'YOUR_TARGET'  # <-- EDIT: target column
X = df.drop(columns=[target_col])
y = df[target_col]

# 2. Split & Scale
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

# 3. Convert to PyTorch Tensors
X_train_t = torch.tensor(X_train_scaled, dtype=torch.float32)
X_test_t  = torch.tensor(X_test_scaled, dtype=torch.float32)
y_train_t = torch.tensor(y_train.values, dtype=torch.float32).reshape(-1, 1)
y_test_t  = torch.tensor(y_test.values, dtype=torch.float32).reshape(-1, 1)

print("Input features (input_dim):", X_train_t.shape[1])

Choose Your Model Architecture

In [ ]:
input_dim = X_train_t.shape[1]

# --- OPTION A: Binary Classification (0 or 1, like Assignment 9) ---
class PyTorchBinaryModel(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(input_dim, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
            nn.Sigmoid()  # Squashes output between 0 and 1
        )
    def forward(self, x):
        return self.layers(x)

# --- OPTION B: Regression (Continuous Number, like Assignment 10) ---
class PyTorchRegressionModel(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1)  # No activation at the end for regression
        )
    def forward(self, x):
        return self.layers(x)

Initialise, Train & Print Loss

In [ ]:
# 1. Pick matching Model + Loss Function
model = PyTorchBinaryModel(input_dim)       # <-- EDIT: swap to PyTorchRegressionModel if regression
criterion = nn.BCELoss()                    # <-- EDIT: swap to nn.MSELoss() if regression
optimizer = optim.Adam(model.parameters(), lr=0.001)

# 2. Training Loop
epochs = 1000
for epoch in range(epochs):
    model.train()
    optimizer.zero_grad()
    outputs = model(X_train_t)
    loss = criterion(outputs, y_train_t)
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 100 == 0:
        print(f'Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}')

Evaluate on Test Set

In [ ]:
model.eval()
with torch.no_grad():
    test_outputs = model(X_test_t)
    test_loss = criterion(test_outputs, y_test_t)
    print(f"Test Loss: {test_loss.item():.4f}")

    # --- IF BINARY CLASSIFICATION: Convert probabilities to 0/1 & show Confusion Matrix ---
    preds = (test_outputs >= 0.5).float()
    print(f"Test Accuracy: {accuracy_score(y_test_t, preds):.4f}")
    ConfusionMatrixDisplay.from_predictions(y_test_t, preds)
    plt.show()

    # --- IF REGRESSION: Show Predicted vs Actual Table ---
    # results_df = pd.DataFrame({'Predicted': test_outputs.numpy().flatten(), 'Actual': y_test_t.numpy().flatten()})
    # display(results_df.head(10))